# Задание

1. взять spaceship titanic из kaggle
2. добавить фичи
3. научиться верно предсказывать


In [10]:
# %pip install pandas numpy plotly

In [11]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import abc as abc
import sklearn as skl

# Загрузка дата сета

In [12]:
df = pd.read_csv('./spaceship-titanic/data.csv')


df_clean = df.copy()
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8693 entries, 0 to 8692
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   PassengerId   8693 non-null   object 
 1   HomePlanet    8492 non-null   object 
 2   CryoSleep     8476 non-null   object 
 3   Cabin         8494 non-null   object 
 4   Destination   8511 non-null   object 
 5   Age           8514 non-null   float64
 6   VIP           8490 non-null   object 
 7   RoomService   8512 non-null   float64
 8   FoodCourt     8510 non-null   float64
 9   ShoppingMall  8485 non-null   float64
 10  Spa           8510 non-null   float64
 11  VRDeck        8505 non-null   float64
 12  Name          8493 non-null   object 
 13  Transported   8693 non-null   bool   
dtypes: bool(1), float64(6), object(7)
memory usage: 891.5+ KB


In [13]:
df.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


* PassengerId - A unique Id for each passenger. Each Id takes the form gggg_pp where gggg indicates a group the passenger is travelling with and pp is their number within the group. People in a group are often family members, but not always.
* HomePlanet - The planet the passenger departed from, typically their planet of permanent residence.
* CryoSleep - Indicates whether the passenger elected to be put into suspended animation for the duration of the voyage. Passengers in cryosleep are confined to their cabins.
* Cabin - The cabin number where the passenger is staying. Takes the form deck/num/side, where side can be either P for Port or S for Starboard.
* Destination - The planet the passenger will be debarking to.
* Age - The age of the passenger.
* VIP - Whether the passenger has paid for special VIP service during the voyage. 
* RoomService, FoodCourt, ShoppingMall, Spa, VRDeck - Amount the passenger has billed at each of the Spaceship Titanic's many luxury amenities.
* Name - The first and last names of the passenger.

Target:
* Transported - Whether the passenger was transported to another dimension. This is the target, the column you are trying to predict.

# Исследовательский анализ данных (EDA)

In [14]:

print("Пропуски в данных:")
print(df.isnull().sum())
print(f"\nПроцент пропусков:")
print((df.isnull().sum() / len(df) * 100).round(2))

print()
print(df.describe())

Пропуски в данных:
PassengerId       0
HomePlanet      201
CryoSleep       217
Cabin           199
Destination     182
Age             179
VIP             203
RoomService     181
FoodCourt       183
ShoppingMall    208
Spa             183
VRDeck          188
Name            200
Transported       0
dtype: int64

Процент пропусков:
PassengerId     0.00
HomePlanet      2.31
CryoSleep       2.50
Cabin           2.29
Destination     2.09
Age             2.06
VIP             2.34
RoomService     2.08
FoodCourt       2.11
ShoppingMall    2.39
Spa             2.11
VRDeck          2.16
Name            2.30
Transported     0.00
dtype: float64

               Age   RoomService     FoodCourt  ShoppingMall           Spa  \
count  8514.000000   8512.000000   8510.000000   8485.000000   8510.000000   
mean     28.827930    224.687617    458.077203    173.729169    311.138778   
std      14.489021    666.717663   1611.489240    604.696458   1136.705535   
min       0.000000      0.000000      0.000000

# Feature Engineering

In [ ]:
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

X = df_clean.copy()
y = X.pop('Transported')

X['Deck'] = X['Cabin'].str[0]
X['Num'] = X['Cabin'].str.split('/').str[1].astype('float64', errors='ignore')
X['Side'] = X['Cabin'].str.split('/').str[2]

spending_cols = ['RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']
X['TotalSpent'] = X[spending_cols].fillna(0).sum(axis=1)

X['HasSpent'] = (X['TotalSpent'] > 0).astype(int)

X['Age'] = X['Age'].fillna(X['Age'].median())
X['HomePlanet'] = X['HomePlanet'].fillna('Unknown')
X['CryoSleep'] = X['CryoSleep'].fillna('Unknown')
X['Destination'] = X['Destination'].fillna('Unknown')
X['VIP'] = X['VIP'].fillna('Unknown')
X['Deck'] = X['Deck'].fillna('Unknown')
X['Side'] = X['Side'].fillna('Unknown')

X['CryoSleep'] = X['CryoSleep'].map({'True': 1, 'False': 0, 'Unknown': -1, True: 1, False: 0})
X['VIP'] = X['VIP'].map({'True': 1, 'False': 0, 'Unknown': -1, True: 1, False: 0})

le_dict = {}
categorical_cols = ['HomePlanet', 'Destination', 'Deck', 'Side']

for col in categorical_cols:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].astype(str))
    le_dict[col] = le

X = X.drop(['PassengerId', 'Cabin', 'Name', 'Num'], axis=1)

# 
print("Новые признаки созданы!")
print(f"Форма данных: {X.shape}")
print(f"\nКолонки: {list(X.columns)}")

Новые признаки созданы!
Форма данных: (8693, 14)

Колонки: ['HomePlanet', 'CryoSleep', 'Destination', 'Age', 'VIP', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck', 'Deck', 'Side', 'TotalSpent', 'HasSpent']


# Обучение и оценка моделей

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Размер обучающего набора: {X_train.shape}")
print(f"Размер тестового набора: {X_test.shape}")
print(f"\nРаспределение классов в обучающем наборе:")
print(y_train.value_counts())

print("\nОбучение модели Random Forest...")
rf_model = RandomForestClassifier(n_estimators=100, max_depth=15, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)

y_pred = rf_model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print(f"\nТочность на тестовом наборе: {accuracy:.4f}")
print(f"\nОтчёт о классификации:")
print(classification_report(y_test, y_pred))

Размер обучающего набора: (6954, 14)
Размер тестового набора: (1739, 14)

Распределение классов в обучающем наборе:
Transported
True     3502
False    3452
Name: count, dtype: int64

Обучение модели Random Forest...

Точность на тестовом наборе: 0.7982

Отчёт о классификации:
              precision    recall  f1-score   support

       False       0.78      0.82      0.80       863
        True       0.82      0.78      0.79       876

    accuracy                           0.80      1739
   macro avg       0.80      0.80      0.80      1739
weighted avg       0.80      0.80      0.80      1739

